In [4]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests


In [ ]:
# Paths for downloads
def resolve_project_root() -> Path:
    # Support running the notebook from the project root or from ./scripts
    cwd = Path.cwd()
    if (cwd / 'data').exists():
        return cwd
    if cwd.name == 'scripts' and (cwd.parent / 'data').exists():
        return cwd.parent
    return cwd

PROJECT_ROOT = resolve_project_root()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'

PATH_GDP = DATA_RAW / 'gdp'
PATH_INFLATION = DATA_RAW / 'inflation'
PATH_EMPLOYMENT = DATA_RAW / 'employment'
PATH_DEBT = DATA_RAW / 'debt'
PATH_COUNTRIES = DATA_RAW / 'countries'

# Create target folders if they do not exist
for p in [PATH_GDP, PATH_INFLATION, PATH_EMPLOYMENT, PATH_DEBT, PATH_COUNTRIES]:
    p.mkdir(parents=True, exist_ok=True)

# API data selection (IMF DataMapper)
url_gdp_pc = 'https://www.imf.org/external/datamapper/api/v1/NGDPDPC'
url_gdp_ppp_world = 'https://www.imf.org/external/datamapper/api/v1/PPPSH'
url_gdp_pc_ppp = 'https://www.imf.org/external/datamapper/api/v1/PPPPC'
url_unemployment = 'https://www.imf.org/external/datamapper/api/v1/LUR'
url_debt = 'https://www.imf.org/external/datamapper/api/v1/GGXWDG_NGDP'
url_inflation = 'https://www.imf.org/external/datamapper/api/v1/PCPIECH'
url_countries = 'https://www.imf.org/external/datamapper/api/v1/countries'

# Map dataset names to output folders and URLs
endpoints = {
    'gdp_per_capita_usd': (PATH_GDP, url_gdp_pc),
    'gdp_ppp_world_share': (PATH_GDP, url_gdp_ppp_world),
    'gdp_per_capita_ppp': (PATH_GDP, url_gdp_pc_ppp),
    'unemployment_rate': (PATH_EMPLOYMENT, url_unemployment),
    'public_debt_gdp': (PATH_DEBT, url_debt),
    'inflation_avg_consumer': (PATH_INFLATION, url_inflation),
    'countries': (PATH_COUNTRIES, url_countries),
}


In [6]:
def fetch_json(url: str) -> dict:
    # Fetch JSON content from an API endpoint
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()


def write_json(path: Path, data: dict) -> None:
    # Write JSON to disk with indentation for readability
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2))


def format_bytes(num: int) -> str:
    # Human-friendly file size formatting
    value = float(num)
    for unit in ['B', 'KiB', 'MiB', 'GiB']:
        if value < 1024 or unit == 'GiB':
            if unit == 'B':
                return f'{int(value)} {unit}'
            return f'{value:.1f} {unit}'
        value /= 1024
    return f'{value:.1f} GiB'


log_lines = []
timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

# Download each endpoint and store its JSON output
for name, (folder, url) in endpoints.items():
    try:
        data = fetch_json(url)
        file_path = folder / f'{name}.json'
        write_json(file_path, data)
        size_bytes = file_path.stat().st_size
        file_type = file_path.suffix.lstrip('.')
        log_lines.append(f'[{timestamp}] {name} | url={url} | file={file_path} | type={file_type} | size={size_bytes} bytes ({format_bytes(size_bytes)})')
    except Exception as exc:
        log_lines.append(f'[{timestamp}] {name} | url={url} | ERROR: {exc}')

# Write a simple metadata log for all downloads
log_path = DATA_RAW / 'api_download_log.txt'
log_path.write_text('\n'.join(log_lines) + '\n')
log_path


PosixPath('/Users/rgctechfi/Projects/ecodata_cloud/data/raw/api_download_log.txt')